# 03 · Limpieza de estimados con la regla original del proyecto

**Entra:** Excel originales. **Sale:** un solo `estimados_semanales_limpio.parquet`.

Se conserva la regla del notebook recuperado `1. Extracción de estimados.ipynb`: elegir un archivo principal no ajustado por año/semana; si falta la semana solicitada, usar el siguiente disponible del mismo año. H1–H5 se calculan desde la **semana solicitada**, no desde la semana del archivo posterior.

Se mantienen separados `origen_solicitado`, `origen_nominal` del archivo, `archivo_es_posterior` y el desfase. Recuperar retrospectivamente un dato no demuestra que se conociera en el origen solicitado: esos casos no se admiten como predictores históricos.

Los ajustes y duplicados originales no se borran del disco; la salida oficial selecciona el archivo principal como hacía la lógica original. La lectura de caché interna Excel se conserva y valida su relación XML. La unidad ESTI. EXPORTABLE requiere contraste con tallos reportados.


## 1. Imports y rutas

In [1]:
from pathlib import Path
import sys
import re
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Datos").is_dir() and (p / "Notebooks").is_dir())
from datetime import date
import unicodedata
from pathlib import PurePosixPath
from zipfile import ZipFile
from lxml import etree

def texto(valor):
    if pd.isna(valor):
        return ""
    s = unicodedata.normalize("NFKD", str(valor).replace("\xa0", " "))
    return re.sub(r"\s+", " ", "".join(c for c in s if not unicodedata.combining(c))).strip().upper()

def normalizar(serie):
    return (serie.astype("string").str.normalize("NFKD")
            .str.replace("[\u0300-\u036f]", "", regex=True)
            .str.replace(r"\s+", " ", regex=True).str.strip().str.upper()
            .replace({"": pd.NA, "NAN": pd.NA, "NONE": pd.NA, "<NA>": pd.NA}))

def finca(serie):
    return normalizar(serie).replace({"GF": "GAITANA", "LA GAITANA": "GAITANA", "¡": "GAITANA",
                                     "AR": "ARABELLA", "LA CABANA": "CABANA", "FINCA LA CABANA": "CABANA"})

def semana_operativa(anio, semana):
    """Lunes representativo. Toda etiqueta 53 se convierte en 01 del año siguiente."""
    y = pd.to_numeric(anio, errors="coerce").astype("Int64")
    w = pd.to_numeric(semana, errors="coerce").astype("Int64")
    y = y + w.eq(53).fillna(False).astype(int)
    w = w.mask(w.eq(53), 1)
    etiquetas = y.astype("string") + "-" + w.astype("string")
    conversion = {}
    for etiqueta in etiquetas.dropna().unique():
        try:
            yy, ww = map(int, etiqueta.split("-"))
            conversion[etiqueta] = pd.Timestamp(date.fromisocalendar(yy, ww, 1)) if 1 <= ww <= 52 else pd.NaT
        except (ValueError, OverflowError):
            conversion[etiqueta] = pd.NaT
    return pd.to_datetime(etiquetas.map(conversion))

def leer_estimado_cache(ruta):
    """Leer una caché con campos de estimado, enlazando definición→records por relación XML.

    No escoger por orden del ZIP; libros con varias cachés requieren una estructura
    reconocible única. Se conserva todo el contenido ESTIMADO de la fuente elegida.
    """
    ns = "http://schemas.openxmlformats.org/spreadsheetml/2006/main"

    def valor(item, compartidos):
        tipo = etree.QName(item).localname
        v = item.get("v")
        if tipo == "m":
            return None
        if tipo == "x":
            return compartidos[int(v)]
        if tipo == "n":
            return float(v)
        if tipo == "b":
            return v == "1"
        return v

    with ZipFile(ruta) as libro:
        candidatos = []
        for nombre in libro.namelist():
            if "pivotcachedefinition" not in nombre.lower() or "/_rels/" in nombre.lower():
                continue
            raiz = etree.fromstring(libro.read(nombre))
            campos = raiz.find(f"{{{ns}}}cacheFields")
            columnas = [texto(c.get("name", "")) for c in campos]
            if {"TIPO", "SEMANA", "ANO", "ESTI. EXPORTABLE"} <= set(columnas):
                fuente = raiz.find(f"{{{ns}}}cacheSource/{{{ns}}}worksheetSource")
                rango = fuente.get("ref", "") if fuente is not None else ""
                if rango:
                    from openpyxl.utils.cell import range_boundaries
                    primera_col, _, ultima_col, _ = range_boundaries(rango)
                    if ultima_col - primera_col + 1 < len(columnas):
                        continue  # Ej.: caché residual Z:Z anuncia diez campos; no es fuente completa.
                candidatos.append((nombre, campos, columnas, float(raiz.get("refreshedDate", "0"))))
        if len(candidatos) > 1:
            ultimo_refresco = max(c[3] for c in candidatos)
            candidatos = [c for c in candidatos if c[3] == ultimo_refresco]
        if len(candidatos) != 1:
            raise ValueError(f"{ruta.name}: {len(candidatos)} cachés de estimado reconocidas; revisar sin inventar datos")
        nombre, campos, columnas, refresco = candidatos[0]
        carpeta = PurePosixPath(nombre).parent
        rel = str(carpeta / "_rels" / (PurePosixPath(nombre).name + ".rels"))
        relaciones = etree.fromstring(libro.read(rel))
        targets = [r.get("Target") for r in relaciones if r.get("Type", "").endswith("/pivotCacheRecords")]
        if len(targets) != 1:
            raise ValueError(f"Relación de records ambigua: {ruta}")
        target = targets[0]
        records = target.lstrip("/") if target.startswith("/") else str(carpeta / target)
        compartidos = []
        for c in campos:
            bloque = c.find(f"{{{ns}}}sharedItems")
            compartidos.append([] if bloque is None else [valor(item, []) for item in bloque])
        pos_tipo = columnas.index("TIPO")
        filas = []
        with libro.open(records) as flujo:
            for _, r in etree.iterparse(flujo, events=("end",), tag=f"{{{ns}}}r"):
                celdas = list(r)
                tipo = valor(celdas[pos_tipo], compartidos[pos_tipo]) if len(celdas) > pos_tipo else None
                if texto(tipo) == "ESTIMADO":
                    filas.append([valor(celdas[i], compartidos[i]) if i < len(celdas) else None for i in range(len(columnas))])
                r.clear()
                while r.getprevious() is not None:
                    del r.getparent()[0]
    resultado = pd.DataFrame(filas, columns=columnas)
    resultado.attrs["cache_origen"] = nombre
    resultado.attrs["refresco_cache_excel"] = refresco  # NO fecha de generación del pronóstico.
    return resultado

SALIDA = ROOT / "Datos_analiticos_proyecto"
SALIDA.mkdir(exist_ok=True)
pd.set_option("display.max_columns", 35)
print("Fuentes originales:", ROOT / "Datos")
print("Salida oficial:", SALIDA)

ENTRADA = ROOT / "Datos/Estimados semanales"
ARCHIVO_SALIDA = SALIDA / "estimados_semanales_limpio.parquet"
archivos = sorted(p for p in ENTRADA.glob("*/*.xlsx") if re.fullmatch(r"20\d{2}",p.parent.name))
filas = []
for p in archivos:
    match = re.search(r"SEM(?:ANA)?\s*[-_ ]*(\d{1,2})",texto(p.stem))
    anios_nombre = re.findall(r"(?<!\d)(20\d{2})(?!\d)", p.stem)
    anio = int(anios_nombre[0]) if anios_nombre else int(p.parent.name)
    filas.append({"archivo":str(p.relative_to(ENTRADA)),"anio":anio,
                  "semana":int(match.group(1)) if match else pd.NA,
                  "ajustado":bool(re.search(r"AJUST|CORREG|CORRECC",texto(p.stem))),
                  "envio":"ENVIO" in texto(p.stem)})
inventario = pd.DataFrame(filas)
display(inventario.groupby(["anio","ajustado"],dropna=False).size().to_frame("archivos"))
assert inventario.semana.notna().all(), "Resolver nombres de archivo no reconocidos antes de seguir"


Fuentes originales: C:\Materias y Cosas Técnicas Uniandes\Proyecto de grado\Proyecto de grado 2\Datos
Salida oficial: C:\Materias y Cosas Técnicas Uniandes\Proyecto de grado\Proyecto de grado 2\Datos_analiticos_proyecto


archivos
anio ajustado          
2023 False           48
2024 False           49
     True             1
2025 False           47
2026 False           36
     True             3

## 2. Estructura real de la fuente

La utilidad de lectura enlaza definición y registros de la caché mediante la relación interna del Excel; no escoge una tabla al azar. Sólo lee registros TIPO=ESTIMADO. Las transformaciones analíticas se muestran abajo.

In [2]:
ejemplo = leer_estimado_cache(archivos[0])
display(ejemplo.head())
print(ejemplo.columns.tolist())


,CODFINCA,PRODUCTO,COLOR,VARIEDAD,TIPO,SUPERVISOR,ANO,SEMANA,ESTI. EXPORTABLE,VARIEDADES VALIDAS
0,GF,Raffine,bicolor burgundy,Petit Faye,ESTIMADO,None,2022.0,52.0,11060.0,incluir
1,GF,Raffine,bicolor purple,Milou,ESTIMADO,None,2022.0,52.0,9000.0,incluir
2,GF,Raffine,bicolor purple,Leo,ESTIMADO,None,2022.0,52.0,9200.0,incluir
3,GF,Raffine,bicolor purple,Forever,ESTIMADO,None,2022.0,52.0,12200.0,incluir
4,GF,Raffine,light pink,Thia,ESTIMADO,None,2022.0,52.0,9000.0,incluir


['CODFINCA', 'PRODUCTO', 'COLOR', 'VARIEDAD', 'TIPO', 'SUPERVISOR', 'ANO', 'SEMANA', 'ESTI. EXPORTABLE', 'VARIEDADES VALIDAS']


## 3. Leer los registros originales una vez por libro

Los 184 libros se leen de sus fuentes, antes de seleccionar las solicitudes. Se utilizan cuatro lectores como en el proceso original para no repetir la lectura del mismo libro por cada semana faltante. No se generan archivos intermedios. La semana 53 pasa a 01 del año siguiente tanto en el archivo como en el objetivo; las etiquetas fuente se conservan.


In [3]:
from concurrent.futures import ThreadPoolExecutor

def leer_y_normalizar(fila):
    archivo = ENTRADA / fila.archivo
    df = leer_estimado_cache(archivo)
    df["fila_cache"] = np.arange(1,len(df)+1)
    df["cache_origen"] = df.attrs["cache_origen"]
    df["refresco_cache_excel"] = df.attrs["refresco_cache_excel"]
    origen = semana_operativa(pd.Series([fila.anio]),pd.Series([fila.semana])).iloc[0]
    df["semana_inicio"] = semana_operativa(df["ANO"],df["SEMANA"])
    df["origen_nominal"] = origen
    df["horizonte_nominal"] = ((df.semana_inicio-origen).dt.days/7+1).astype("Int64")
    df["finca"] = finca(df["CODFINCA"])
    for col in ["PRODUCTO","COLOR","VARIEDAD"]:
        df[col.lower()] = normalizar(df[col])
    df["estimado"] = pd.to_numeric(df["ESTI. EXPORTABLE"],errors="coerce")
    df["anio_objetivo_fuente"] = pd.to_numeric(df["ANO"],errors="coerce").astype("Int64")
    df["semana_objetivo_fuente"] = pd.to_numeric(df["SEMANA"],errors="coerce").astype("Int64")
    df["archivo_origen"] = str(archivo.relative_to(ROOT)).replace("\\","/")
    df["es_ajustado"],df["marca_envio"] = fila.ajustado,fila.envio
    df["fecha_generacion_confirmada"] = False
    df["unidad"] = "tallos ESTI. EXPORTABLE; equivalencia con reportados por validar"
    # Conservar las demás dimensiones originales como texto evita perder supervisor/tipo.
    cols = ["finca","producto","color","variedad","semana_inicio","origen_nominal","horizonte_nominal",
            "estimado","anio_objetivo_fuente","semana_objetivo_fuente","archivo_origen","fila_cache",
            "es_ajustado","marca_envio","fecha_generacion_confirmada","unidad","cache_origen","refresco_cache_excel"]
    extra = [c for c in ["SUPERVISOR","TIPO FLOR","TIPO_FLOR","CODFINCA"] if c in df]
    for c in extra: df[c] = df[c].astype("string")
    control = {"archivo":fila.archivo,"filas":len(df),"h1_h5":int(df.horizonte_nominal.between(1,5).sum()),"sin_cantidad":int(df.estimado.isna().sum())}
    return df[cols+extra], control

partes, controles = [], []
with ThreadPoolExecutor(max_workers=4) as lectores:
    for parte,control in lectores.map(leer_y_normalizar,inventario.itertuples(index=False)):
        partes.append(parte)
        controles.append(control)
        if len(controles)%20==0:print(f"Leídos {len(controles)}/{len(inventario)} libros",flush=True)
estimados = pd.concat(partes,ignore_index=True)
del partes, parte
display(pd.DataFrame(controles).head())


Leídos 20/184 libros


Leídos 40/184 libros


Leídos 60/184 libros


Leídos 80/184 libros


Leídos 100/184 libros


Leídos 120/184 libros


Leídos 140/184 libros


Leídos 160/184 libros


Leídos 180/184 libros


,archivo,filas,h1_h5,sin_cantidad
0,2023\Estimado Sem 01- 07 2023 Gaitana-Arabel...,1271,805,0
1,2023\Estimado Sem 02- 07 2023 Gaitana-Arabel...,1110,805,0
2,2023\Estimado Sem 03- 07 2023 Gaitana-Arabel...,954,793,0
3,2023\Estimado Sem 05- 09 2023 Gaitana-Arabel...,973,810,0
4,2023\Estimado Sem 06- 10 2023 Gaitana-Arabel...,969,805,0


## 4. Archivo principal, semana faltante y H1–H5

La prioridad reproduce el original: archivos no ajustados; preferir marca ENVIO; penalizar sello de descarga; desempatar por nombre. Para cada año se solicitan semanas 1 a la mayor semana disponible. Se usa la exacta o la siguiente, nunca una anterior ni otro año. El ejemplo 2026-S02 debe usar S03.

Una misma línea fuente puede atender varias solicitudes: es reutilización explícita del estimado, no producción adicional. La llave final es archivo/fila_cache/origen_solicitado. El horizonte solicitado queda en `horizonte`; `horizonte_nominal` conserva la distancia respecto al archivo realmente utilizado.

No se inventan cantidades si la caché no contiene una semana objetivo. Los controles de solicitudes sin filas se muestran aquí, sin exportar otro CSV.

Se prioriza el a?o del nombre sobre la carpeta: el archivo 2024-S01 est? bajo 2023. La lectura integral de los 184 libros se ejecut?; despu?s se corrigieron inventario y selecci?n reutilizando esa lectura ya disponible en memoria, sin generar otro dataset. Ejecutar todo vuelve a leer los originales con esta regla corregida.


In [4]:
# El nombre identifica el a?o; una carpeta equivocada no cambia la emisi?n.
identidad_archivo = inventario.copy()
identidad_archivo["archivo_origen"] = identidad_archivo.archivo.map(
    lambda x: str((ENTRADA/x).relative_to(ROOT)).replace("\\", "/"))
identidad_archivo["origen_archivo"] = semana_operativa(identidad_archivo.anio, identidad_archivo.semana)
assert not identidad_archivo.duplicated("archivo_origen").any()
estimados["origen_nominal"] = estimados.archivo_origen.map(
    identidad_archivo.set_index("archivo_origen").origen_archivo)
assert estimados.origen_nominal.notna().all()
estimados["horizonte_nominal"] = ((estimados.semana_inicio-estimados.origen_nominal).dt.days/7+1).astype("Int64")

# Selección principal exactamente como el notebook original.
candidatos = inventario.loc[~inventario.ajustado].copy()
candidatos["sello_horario"] = candidatos.archivo.str.contains(r"\d{8}[_ -]\d{6}",regex=True)
candidatos["prioridad"] = candidatos.envio.astype(int)*-2+candidatos.sello_horario.astype(int)*2
seleccionados = candidatos.sort_values(["anio","semana","prioridad","archivo"]).drop_duplicates(["anio","semana"])
solicitudes = []
for anio,g in seleccionados.groupby("anio"):
    g = g.sort_values("semana")
    for semana in range(1,int(g.semana.max())+1):
        f = g.loc[g.semana.ge(semana)].iloc[0]
        solicitudes.append({"anio_solicitado":int(anio),"semana_solicitada":semana,
            "anio_archivo":int(anio),"semana_archivo":int(f.semana),
            "archivo_origen":str((ENTRADA/f.archivo).relative_to(ROOT)).replace("\\","/"),
            "archivo_es_posterior":int(f.semana)>semana,
            "diferencia_semanas_archivo":int(f.semana)-semana})
plan = pd.DataFrame(solicitudes)
plan["origen_solicitado"] = semana_operativa(plan.anio_solicitado,plan.semana_solicitada)
assert not plan.duplicated("origen_solicitado").any(),"Revisar colisión semana 53/01 antes de continuar"
assert not estimados.duplicated(["archivo_origen","fila_cache"]).any()
originales_en_memoria = estimados
partes_solicitadas = []
for f in plan.itertuples(index=False):
    bloque = originales_en_memoria.loc[originales_en_memoria.archivo_origen.eq(f.archivo_origen)].copy()
    bloque["horizonte"] = ((bloque.semana_inicio-f.origen_solicitado).dt.days/7+1).astype("Int64")
    bloque = bloque.loc[bloque.horizonte.between(1,5)].copy()
    for col in plan.columns:
        if col!="archivo_origen":bloque[col]=getattr(f,col)
    partes_solicitadas.append(bloque)
estimados = pd.concat(partes_solicitadas,ignore_index=True)
del originales_en_memoria, partes_solicitadas
estimados["en_h1_h5"] = estimados.horizonte.between(1,5)
estimados["disponible_al_origen_confirmado"] = False
llave = ["origen_solicitado","archivo_origen","finca","producto","color","variedad","semana_inicio"]
estimados["llave_comercial_repetida"] = estimados.duplicated(llave,keep=False)
assert not estimados.duplicated(["archivo_origen","fila_cache","origen_solicitado"]).any()
assert estimados.en_h1_h5.all()
assert estimados.origen_nominal.ge(estimados.origen_solicitado).all()
assert estimados.archivo_es_posterior.eq(estimados.origen_nominal.gt(estimados.origen_solicitado)).all()
control_solicitudes = plan.merge(estimados.groupby("origen_solicitado").agg(filas=("estimado","size"),horizontes=("horizonte","nunique")),on="origen_solicitado",how="left",validate="one_to_one")
display(control_solicitudes)
display(control_solicitudes.loc[control_solicitudes.archivo_es_posterior])
ejemplo_s02 = control_solicitudes.query("anio_solicitado==2026 and semana_solicitada==2")
assert len(ejemplo_s02)==1 and ejemplo_s02.semana_archivo.iloc[0]==3
assert ejemplo_s02.filas.iloc[0]>0,"La regla debe recuperar datos, no sólo un nombre de archivo"
display(estimados.query("anio_solicitado==2026 and semana_solicitada==2")[["origen_solicitado","origen_nominal","semana_inicio","horizonte","finca","variedad","estimado","archivo_es_posterior"]].head(10))
print("Libros originales leídos:",len(inventario),"Principales:",len(seleccionados),"Solicitudes:",len(plan))
print("Solicitudes recuperadas:",int(plan.archivo_es_posterior.sum()),"Filas finales:",len(estimados))
display(estimados.groupby("finca",dropna=False).agg(filas=("estimado","size"),desde=("semana_inicio","min"),hasta=("semana_inicio","max")))


,anio_solicitado,semana_solicitada,anio_archivo,semana_archivo,archivo_origen,archivo_es_posterior,diferencia_semanas_archivo,origen_solicitado,filas,horizontes
0,2023,1,2023,1,Datos/Estimados semanales/2023/Estimado Sem 0...,False,0,2023-01-02,805,5
1,2023,2,2023,2,Datos/Estimados semanales/2023/Estimado Sem 0...,False,0,2023-01-09,805,5
2,2023,3,2023,3,Datos/Estimados semanales/2023/Estimado Sem 0...,False,0,2023-01-16,793,5
3,2023,4,2023,5,Datos/Estimados semanales/2023/Estimado Sem 0...,True,1,2023-01-23,813,5
4,2023,5,2023,5,Datos/Estimados semanales/2023/Estimado Sem 0...,False,0,2023-01-30,810,5
...,...,...,...,...,...,...,...,...,...,...
188,2026,33,2026,33,Datos/Estimados semanales/2026/Estimado sem 33...,False,0,2026-08-10,1021,5
189,2026,34,2026,34,Datos/Estimados semanales/2026/Estimado sem 34...,False,0,2026-08-17,1025,5
190,2026,35,2026,35,Datos/Estimados semanales/2026/Estimado sem 35...,False,0,2026-08-24,1023,5
191,2026,36,2026,36,Datos/Estimados semanales/2026/Estimado sem 36...,False,0,2026-08-31,1021,5


,anio_solicitado,semana_solicitada,anio_archivo,semana_archivo,archivo_origen,archivo_es_posterior,diferencia_semanas_archivo,origen_solicitado,filas,horizontes
3,2023,4,2023,5,Datos/Estimados semanales/2023/Estimado Sem 0...,True,1,2023-01-23,813,5
14,2023,15,2023,16,Datos/Estimados semanales/2023/Estimado Sem 1...,True,1,2023-04-10,814,5
31,2023,32,2023,33,Datos/Estimados semanales/2023/Estimado Sem 3...,True,1,2023-08-07,819,5
41,2023,42,2023,43,Datos/Estimados semanales/2023/Estimado Sem 4...,True,1,2023-10-16,896,5
44,2023,45,2023,46,Datos/Estimados semanales/2023/Estimado Sem 4...,True,1,2023-11-06,895,5
75,2024,24,2024,26,Datos/Estimados semanales/2024/estimado sem 26...,True,2,2024-06-10,808,4
76,2024,25,2024,26,Datos/Estimados semanales/2024/estimado sem 26...,True,1,2024-06-17,1010,5
84,2024,33,2024,34,Datos/Estimados semanales/2024/Estimado Sem 34...,True,1,2024-08-12,1025,5
136,2025,33,2025,35,Datos/Estimados semanales/2025/estimado sem 35...,True,2,2025-08-11,722,4
137,2025,34,2025,35,Datos/Estimados semanales/2025/estimado sem 35...,True,1,2025-08-18,902,5


,origen_solicitado,origen_nominal,semana_inicio,horizonte,finca,variedad,estimado,archivo_es_posterior
140734,2026-01-05,2026-01-12,2026-01-05,1,CABANA,ALPINIUM,14500.0,True
140735,2026-01-05,2026-01-12,2026-01-05,1,CABANA,TROPICAL SNOW,900.0,True
140736,2026-01-05,2026-01-12,2026-01-12,2,CABANA,ALPINIUM,14350.0,True
140737,2026-01-05,2026-01-12,2026-01-12,2,CABANA,TROPICAL SNOW,900.0,True
140738,2026-01-05,2026-01-12,2026-01-19,3,CABANA,ALPINIUM,14500.0,True
140739,2026-01-05,2026-01-12,2026-01-19,3,CABANA,TROPICAL SNOW,900.0,True
140740,2026-01-05,2026-01-12,2026-01-05,1,GAITANA,ANTIGUA,2840.0,True
140741,2026-01-05,2026-01-12,2026-01-12,2,GAITANA,ANTIGUA,3280.0,True
140742,2026-01-05,2026-01-12,2026-01-19,3,GAITANA,ANTIGUA,3130.0,True
140743,2026-01-05,2026-01-12,2026-01-26,4,GAITANA,ANTIGUA,3120.0,True


Libros originales leídos: 184 Principales: 178 Solicitudes: 193
Solicitudes recuperadas: 15 Filas finales: 176140


,filas,desde,hasta
finca,,,
ARABELLA,48687,2023-01-02,2026-10-05
CABANA,3884,2023-01-02,2026-10-05
GAITANA,123569,2023-01-02,2026-10-05


## 5. Exportación única y resumen

El notebook no calcula errores ni entrena modelos. Al reejecutar reemplaza la misma salida; no crea H1 separado ni archivos por emisión.

In [5]:
estimados.to_parquet(ARCHIVO_SALIDA,index=False)
print("EXPORTADO:",ARCHIVO_SALIDA,estimados.shape)
display(estimados.head())


EXPORTADO: C:\Materias y Cosas Técnicas Uniandes\Proyecto de grado\Proyecto de grado 2\Datos_analiticos_proyecto\estimados_semanales_limpio.parquet (176140, 31)


,finca,producto,color,variedad,semana_inicio,origen_nominal,horizonte_nominal,estimado,anio_objetivo_fuente,semana_objetivo_fuente,archivo_origen,fila_cache,es_ajustado,marca_envio,fecha_generacion_confirmada,unidad,cache_origen,refresco_cache_excel,SUPERVISOR,CODFINCA,llave_comercial_repetida,en_h1_h5,horizonte,anio_solicitado,semana_solicitada,anio_archivo,semana_archivo,archivo_es_posterior,diferencia_semanas_archivo,origen_solicitado,disponible_al_origen_confirmado
0,GAITANA,RAFFINE,BICOLOR BURGUNDY,PETIT FAYE,2023-01-02,2023-01-02,1,10800.0,2023,1,Datos/Estimados semanales/2023/Estimado Sem 0...,7,False,False,False,tallos ESTI. EXPORTABLE; equivalencia con repo...,xl/pivotCache/pivotCacheDefinition1.xml,44925.267271,<NA>,GF,False,True,1,2023,1,2023,1,False,0,2023-01-02,False
1,GAITANA,RAFFINE,BICOLOR PURPLE,MILOU,2023-01-02,2023-01-02,1,9340.0,2023,1,Datos/Estimados semanales/2023/Estimado Sem 0...,8,False,False,False,tallos ESTI. EXPORTABLE; equivalencia con repo...,xl/pivotCache/pivotCacheDefinition1.xml,44925.267271,<NA>,GF,False,True,1,2023,1,2023,1,False,0,2023-01-02,False
2,GAITANA,RAFFINE,BICOLOR PURPLE,LEO,2023-01-02,2023-01-02,1,9110.0,2023,1,Datos/Estimados semanales/2023/Estimado Sem 0...,9,False,False,False,tallos ESTI. EXPORTABLE; equivalencia con repo...,xl/pivotCache/pivotCacheDefinition1.xml,44925.267271,<NA>,GF,False,True,1,2023,1,2023,1,False,0,2023-01-02,False
3,GAITANA,RAFFINE,BICOLOR PURPLE,FOREVER,2023-01-02,2023-01-02,1,12250.0,2023,1,Datos/Estimados semanales/2023/Estimado Sem 0...,10,False,False,False,tallos ESTI. EXPORTABLE; equivalencia con repo...,xl/pivotCache/pivotCacheDefinition1.xml,44925.267271,<NA>,GF,False,True,1,2023,1,2023,1,False,0,2023-01-02,False
4,GAITANA,RAFFINE,LIGHT PINK,THIA,2023-01-02,2023-01-02,1,9000.0,2023,1,Datos/Estimados semanales/2023/Estimado Sem 0...,11,False,False,False,tallos ESTI. EXPORTABLE; equivalencia con repo...,xl/pivotCache/pivotCacheDefinition1.xml,44925.267271,<NA>,GF,False,True,1,2023,1,2023,1,False,0,2023-01-02,False
